In [ ]:
# Transformer 모델 구축 - Transformer RAG(Retriever Augmentation Generation) 구성 및 모델 파이프라인 구축(Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: FAISS 메모리 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용 
# 1. DB에서 수집데이터 조회 + 로깅 설정
# - PostgreSQL 서버: D:\AI\PostgreSQL\18\bin>pg_ctl start -D "D:\AI\PostgreSQL\18\data"
# - PostgreSQL 서버 포트 확인: netstat -ano | findstr 5432
# - DB에서 수집데이터 조회: 관계형 DB, PostgreSql
# 2. Qdrant 의미 기반 검색엔진 
# - 서버 + 클라이언트 연결
# - 컬렉션 생성: news_collection
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections
# 3. 임베딩 생성 + Qdrant 저장
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert
# 4. Qdrant collection 정보 확인
# - 전체 조회 + 특정 ID 조회
# 5. Qdrant collection 데이터 삭제
# - 특정 ID 삭제 + 조건 기반 데이터 삭제
# 6. Qdrant collection 데이터 삭제: 
# - 컬렉션 유지 + 전체 데이터 삭제
# 7. Qdrant 의미 기반 검색
# - Qdrant 의미 기반 검색엔진을 통한 검색 결과 도출
# 8. Qdrant 의미 기반 검색 -> QA 토크나이저/QA 모델
# - 모델: monologg/koelectra-base-v3-finetuned-korquad
# - 특정 도메인으로 파인튜닝된 QA 모델로 교체 검토
# 9. Qdrant 의미 기반 검색 + QA 모델 결과 -> 요약 토크나이저/요약 모델
# - 모델: gogamza/kobart-summarization
# - tokenizer: PreTrainedTokenizerFast 적용, BartTokenizer 영어 전용이라 맞지 않음
# - 특정 도메인으로 파인튜닝된 QA 모델로 교체 검토
# 10. 요약 모델 결과 -> LLM 토크나이저/LLM 모델
# - 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능
# - 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

In [7]:
# DB에서 수집데이터 조회 + 로깅 설정
import psycopg2
import json
import logging

# 로깅 설정: 파일 저장
logging.basicConfig(
    level=logging.INFO, # 로그 레벨: INFO 이상 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
    handlers=[
        logging.FileHandler("23.transformer_rag2_app.log", encoding="utf-8"), # 로그를 파일에 저장
        logging.StreamHandler() # 콘솔에도 출력
    ]
)
# DB에서 수집데이터 조회
def fetch_data():
    try:
        with psycopg2.connect(
            dbname="newsdb",      # 생성한 DB 이름
            user="newsuser",      # 생성한 사용자
            password="1234",      # 설정한 비밀번호
            host="localhost",     # 로컬에서 실행 중
            port="5432"           # 기본 포트
        ) as conn:
            with conn.cursor() as cur:
                # 모든 컬럼 조회
                cur.execute("SELECT id, title, content, source, date FROM news_view;") # SQL 쿼리 실행
                rows = cur.fetchall() # fetchall() 전체 결과를 가져옴

                # 컬럼 이름 가져오기
                col_names = [ desc[0] for desc in cur.description ]

                # 결과를 JSON 형태로 변환
                result = []
                for row in rows:
                    row_dict = dict(zip(col_names, row)) # 결과를 dict 형태로 변환: 컬럼 이름과 데이터 매핑 처리
                    row_dict["date"] = str(row_dict["date"]) # date 컬럽 -> 문자열 변환
                    result.append(row_dict)
                logging.info("PostgreSQL 연결 및 조회 성공!")
                return result
    except Exception as e:
        logging.error("데이터 조회 중 에러 발생: %s", e)
        return [] # 에러가 발생해도 함수가 항상 빈 리스트를 반환, 프로그램이 멈추지 않고 계속 진행 가능

# DB에서 수집데이터 조회 실행
db_data = fetch_data()
# JSON 변환, ensure_ascii=False 한글 깨짐 방지, indent=4 JSON을 4칸 들여쓰기
print(json.dumps(db_data, ensure_ascii=False, indent=4))

2026-03-16 10:36:06,968 [INFO] PostgreSQL 연결 및 조회 성공!


[
    {
        "id": 21,
        "title": "AI 개요",
        "content": "인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고 있다.",
        "source": "위키백과",
        "date": "2026-03-01"
    },
    {
        "id": 22,
        "title": "중동 전쟁 긴급 회의",
        "content": "유엔은 중동 지역에서 발생한 무력 충돌과 인도적 위기를 해결하기 위해 긴급 회의를 소집하였다. 회의에서는 휴전 협상, 난민 지원, 국제적 평화 유지 활동에 대한 논의가 이루어졌으며, 각국 대표들은 분쟁 해결을 위한 외교적 해법을 모색하였다.",
        "source": "국제뉴스",
        "date": "2026-03-06"
    },
    {
        "id": 23,
        "title": "기후 변화 보고서",
        "content": "IPCC는 최신 보고서를 통해 지구 평균 기온 상승이 예상보다 빠르게 진행되고 있으며, 해수면 상승과 극한 기후 현상이 빈번해지고 있다고 경고했다. 보고서는 각국이 탄소 배출을 줄이고 재생에너지 사용을 확대해야 한다는 점을 강조하였다.",
        "source": "환경뉴스",
        "date": "2026-02-20"
    },
    {
        "id": 24,
        "title": "금융 시장 동향",
        "content": "뉴욕 증시는 기술주 강세와 함께 주요 지수들이 상승세를 보였다. 투자자들은 인공지능과 반도체 산업의 성장 가능성에 주목하고 있으며, 글로벌 경제 회복 기대감이 시장 전반에 긍정

In [17]:
# Qdrant 의미 기반 검색엔진 
# - 서버 + 클라이언트 연결
# - 컬렉션 생성: news_collection
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models # Qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체 클래스) 로드
# Qdrant 클라이언트 연결: Qdrant 검색엔진 서버 실행 및 서버 연결 확인
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant 컬렉션 생성
# - 벡터 크기는 모델에 따라 다르다(sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 -> 384차원)
# - 거리(metric)는 보통 COSINE 사용
# 컬렉션 존재 여부 확인 후 생성
if not qdrant.collection_exists("news_collection"):
    qdrant.create_collection(
        collection_name="news_collection",
        vectors_config=models.VectorParams(
            size=384, # 임베딩 벡터 차원: paraphrase-multilingual-MiniLM-L12-v2 임베딩 모델과 차원을 동일하게 맞추어야 한다
            distance=models.Distance.COSINE # 유사도 계산 방식
        )
    )
    print("Qdrant 컬렉션 생성 완료")
else:
    print("news_collection 컬렉션이 존재합니다.")

2026-03-16 10:50:22,657 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:50:24,720 [INFO] HTTP Request: GET http://localhost:6333/collections/news_collection/exists "HTTP/1.1 200 OK"
2026-03-16 10:50:27,488 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_collection "HTTP/1.1 200 OK"


Qdrant 컬렉션 생성 완료


In [18]:
# 임베딩 생성 + Qdrant 저장
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# 임베딩 모델 로드
# - 여기서는 paraphrase-multilingual-MiniLM-L12-v2 모델을 사용했는데, 다국어(한국어 포함) 문장 의미를 잘 반영하는 임베딩을 생성한다
# - 문장을 입력하면 의미 공간에서 가까운 벡터로 변환
# embedder = SentenceTransformer( # 온라인 상태(단, 로컬캐시에 저장)
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device=device
# )

# 오프라인 사용 준비: 온라인 환경에서 모델 다운로드
# - git lfs install
# - git clone https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedder = SentenceTransformer( # 오프라인 상태(경로 지정에 저장)
    "./offline_models/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)


# 컬렉션에 데이터 삽입
def insert_to_qdrant(data):
    ids = []
    vectors = []
    payloads = []
    for item in data:
        text = item["title"] + " " + item["content"] # 제목+내용 임베딩 대상 데이터
        embedding = embedder.encode(text).tolist() # 제목+내용 임베딩

        ids.append(item["id"]) # ID 값
        vectors.append(embedding) # 벡터 데이터
        payloads.append(item) # 원본 데이터
    
    # qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
    # { - 데이터 저장 형식
    #     "id": 1,
    #     "vector": [0.123, -0.456, 0.789, ...],
    #     "payload": {
    #         "title": "AI 의료 활용",
    #         "content": "AI가 의료 분야에서 활용되는 사례...",
    #         "date": "2026-03-11",
    #         "author": "홍길동"
    #     }
    # }
    qdrant.upsert(
        collection_name="news_collection",
        points=models.Batch(
            ids=ids,
            vectors=vectors,
            payloads=payloads # payloads(JSON) 형태로 저장
        )
    )
    print("데이터 임베딩 및 Qdrant 저장 완료")

# Batch 단위로 Qdrant의 update/insert 테스트
batch_size = 2
for i in range(0, len(db_data), batch_size):
    chunk = db_data[i:i+batch_size] # 2개씩 잘라서 가져옴
    insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

2026-03-16 10:51:55,573 [INFO] Load pretrained SentenceTransformer: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2


PyTorch Version: 2.6.0, Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

데이터 임베딩 및 Qdrant 저장 완료


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-16 10:52:12,576 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_collection/points?wait=true "HTTP/1.1 200 OK"


데이터 임베딩 및 Qdrant 저장 완료


In [19]:
# Qdrant collection 정보 확인: 전체 조회 + 특정 ID 조회
from qdrant_client import QdrantClient

qdrant = QdrantClient(host="localhost", port=6333)

# 컬렉션 정보: 컬렉션 생성시 설정된 정보 및 상태를 보여주는 메타데이터
info = qdrant.get_collection("news_collection")
print(info)

# 전체 데이터 조회: limit 지정
points = qdrant.scroll(
    collection_name="news_collection",
    limit=10 # 최대 10개 반환
)
print(f"전체 데이터 조회: limit 지정\n{points}")

# 특정 ID 데이터 조회
point = qdrant.retrieve(
    collection_name="news_collection",
    ids=[21,22],
    with_payload=True, # payload 정보 포함
    with_vectors=False # vectors 정보 제외
)
print(f"특정 ID 데이터 조회:\n{point}")

2026-03-16 10:52:28,730 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:52:30,777 [INFO] HTTP Request: GET http://localhost:6333/collections/news_collection "HTTP/1.1 200 OK"


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=10 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_config=WalConfi

2026-03-16 10:52:32,826 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points/scroll "HTTP/1.1 200 OK"


전체 데이터 조회: limit 지정
([Record(id=21, payload={'id': 21, 'title': 'AI 개요', 'content': '인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고 있다.', 'source': '위키백과', 'date': '2026-03-01'}, vector=None, shard_key=None, order_value=None), Record(id=22, payload={'id': 22, 'title': '중동 전쟁 긴급 회의', 'content': '유엔은 중동 지역에서 발생한 무력 충돌과 인도적 위기를 해결하기 위해 긴급 회의를 소집하였다. 회의에서는 휴전 협상, 난민 지원, 국제적 평화 유지 활동에 대한 논의가 이루어졌으며, 각국 대표들은 분쟁 해결을 위한 외교적 해법을 모색하였다.', 'source': '국제뉴스', 'date': '2026-03-06'}, vector=None, shard_key=None, order_value=None), Record(id=23, payload={'id': 23, 'title': '기후 변화 보고서', 'content': 'IPCC는 최신 보고서를 통해 지구 평균 기온 상승이 예상보다 빠르게 진행되고 있으며, 해수면 상승과 극한 기후 현상이 빈번해지고 있다고 경고했다. 보고서는 각국이 탄소 배출을 줄이고 재생에너지 사용을 확대해야 한다는 점을 강조하였다.', 'source': '환경뉴스', 'date': '2026-02-20'}, vector=None, shard_key=None, order_value=None), Record(id=24, payload={'id': 24, 'title': '금융 시장 동향', 'content': '뉴욕 증시는 기술주 강세와 함께 주요 지수들이

2026-03-16 10:52:34,913 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points "HTTP/1.1 200 OK"


특정 ID 데이터 조회:
[Record(id=21, payload={'id': 21, 'title': 'AI 개요', 'content': '인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고 있다.', 'source': '위키백과', 'date': '2026-03-01'}, vector=None, shard_key=None, order_value=None), Record(id=22, payload={'id': 22, 'title': '중동 전쟁 긴급 회의', 'content': '유엔은 중동 지역에서 발생한 무력 충돌과 인도적 위기를 해결하기 위해 긴급 회의를 소집하였다. 회의에서는 휴전 협상, 난민 지원, 국제적 평화 유지 활동에 대한 논의가 이루어졌으며, 각국 대표들은 분쟁 해결을 위한 외교적 해법을 모색하였다.', 'source': '국제뉴스', 'date': '2026-03-06'}, vector=None, shard_key=None, order_value=None)]


In [11]:
# Qdrant collection 데이터 삭제: 특정 ID 삭제 + 조건 기반 데이터 삭제
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# 특정 ID 삭제
qdrant.delete(
    collection_name="news_collection",
    points_selector=models.PointIdsList(points=[21]) # delete() 사용하는 ID명은 points 이다
)
print("ID=21 데이터 삭제 완료")

2026-03-16 10:41:17,670 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:41:19,768 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points/delete?wait=true "HTTP/1.1 200 OK"


ID=21 데이터 삭제 완료


In [13]:
# Qdrant collection 데이터 삭제: 전체 데이터 삭제
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# 컬렉션은 유지, 모든 데이터 삭제
qdrant.delete(
    collection_name="news_collection",
    points_selector=models.Filter(
        must=[] # 조건이 없으므로 전체 데이터 삭제
    )
)
print("컬렉션 유지, 전체 데이터 삭제 완료")

2026-03-16 10:41:51,445 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:41:53,533 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points/delete?wait=true "HTTP/1.1 200 OK"


컬렉션 유지, 전체 데이터 삭제 완료


In [15]:
# Qdrant collection 삭제: news_collection 자체 삭제
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# collection 삭제: news_collection 자체 삭제
qdrant.delete_collection(collection_name="news_collection")
print("컬렉션 자체 삭제, news_collection 삭제 완료")

2026-03-16 10:46:33,558 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:46:35,693 [INFO] HTTP Request: DELETE http://localhost:6333/collections/news_collection "HTTP/1.1 200 OK"


컬렉션 자체 삭제, news_collection 삭제 완료


In [20]:
# Qdrant 의미 기반 검색
# - Qdrant 의미 기반 검색엔진을 통한 검색 결과 도출
from qdrant_client import QdrantClient
from qdrant_client.http import models

qdrant = QdrantClient(host="localhost", port=6333)

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# 임베딩 모델 로드
# - 여기서는 paraphrase-multilingual-MiniLM-L12-v2 모델을 사용했는데, 다국어(한국어 포함) 문장 의미를 잘 반영하는 임베딩을 생성한다
# - 문장을 입력하면 의미 공간에서 가까운 벡터로 변환
# embedder = SentenceTransformer( # 온라인 상태(단, 로컬캐시에 저장)
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device=device
# )

# 오프라인 사용 준비: 온라인 환경에서 모델 다운로드
# - git lfs install
# - git clone https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedder = SentenceTransformer( # 오프라인 상태(경로 지정에 저장)
    "./offline_models/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)

query = "AI 의료 활용 사례"
query_vector = embedder.encode(query).tolist()

results = qdrant.query_points(
    collection_name="news_collection",
    query=query_vector,   # 그냥 벡터 리스트를 직접 전달
    limit=3,
    with_payload=True
)

for r in results.points:
    print(f"ID={r.id}, Score={r.score}, Title={r.payload['title']}, Content={r.payload['content']}")

2026-03-16 10:53:34,366 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-03-16 10:53:34,380 [INFO] Load pretrained SentenceTransformer: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2


PyTorch Version: 2.6.0, Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-16 10:53:38,353 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points/query "HTTP/1.1 200 OK"


ID=25, Score=0.78133774, Title=의료 AI 활용, Content=병원에서는 AI 기반 진단 시스템을 도입하여 환자의 영상 데이터를 분석하고 질병을 조기에 발견하는 데 활용하고 있다. 특히 암 진단, 심혈관 질환 예측 등에서 높은 정확도를 보여 의료 현장의 혁신을 이끌고 있다.
ID=21, Score=0.76919353, Title=AI 개요, Content=인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고 있다.
ID=29, Score=0.5149973, Title=기술 혁신 사례, Content=국내 스타트업은 새로운 로봇 기술을 공개하며 제조업과 물류 산업에서의 자동화 가능성을 제시하였다. 이 기술은 인공지능과 센서 융합을 통해 사람과 협업할 수 있는 차세대 로봇을 구현하였다.


In [21]:
# Qdrant 의미 기반 검색 + QA 토크나이저 + QA 모델
# - 문자열 -> 토큰 ID 변환, 예시) "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import pipeline
import gc

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# QA모델: 특정 도메인으로 파인튜닝된 QA 모델로 교체 검토
qa_model_name = "monologg/koelectra-base-v3-finetuned-korquad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name) # KoELECTRA 토크나이저 로드
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name).to(device)
# qa_pipeline = pipeline(
#     "question-answering", # Hugging Face 지정된 task
#     model=qa_model, # Hugging Face 지정된 모델
#     tokenizer=qa_tokenizer # Hugging Face 지정된 토크나이저
# )
def qa_pipeline(question, context):
    # 토크나이징: 질문과 문서를 토큰ID 변환
    inputs = qa_tokenizer(
        question, # 질문
        context, # 문서
        return_tensors="pt"
    ).to(device)

    # 추론 실행
    with torch.no_grad():
        outputs = qa_model(**inputs)

    # 답변 시작/끝 위치 예측
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1

    # 토큰ID -> 토큰 문자열 변환
    tokens = qa_tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0][answer_start:answer_end]
    )
    # 특수 토큰 제거: [CLS],[SEP],[PAD] 특수 토큰이 제외한다
    tokens = [ t for t in tokens if t not in qa_tokenizer.all_special_tokens ]
    # 토큰들을 다시 문자열로 합치기
    answer = qa_tokenizer.convert_tokens_to_string(tokens)
    
    return answer

# Qdrant 의미 기반 검색 결과 문서 가져오기
query = "AI 의료 활용 사례"
query_vector = embedder.encode(query).tolist()
results = qdrant.query_points(
    collection_name="news_collection",
    query=query_vector,   # 그냥 벡터 리스트를 직접 전달
    limit=3,
    with_payload=True
)
for r in results.points:
    print(f"ID={r.id}, Score={r.score}, Title={r.payload['title']}, Content={r.payload['content']}")

# 검색된 문서들을 context로 합치기
contexts = [ r.payload["content"] for r in results.points ] # [문서1, 문서2...]
context = " ".join(contexts) # 여러 문서들을 하나의 문자열로 합침, [문서1, 문서2...] -> 문서1 문서2 ... 변경

# QA 모델 실행: 질문 question 문서 context, 입력으로 받아서 답변을 복원
# - 모델 결과 구조 {'score': 0.12894503772258759, 'start': 12, 'end': 36, 'answer': '진단 보조, 신약 개발, 환자 맞춤형 치료에'}
answer = qa_pipeline(question=query, context=context)

# QA 결과가 비정상일때 fallback 처리
if isinstance(answer, dict): # answer가 dict 타입 확인
    # answer 키에 들어 있는 답변 꺼낸다, 키가 없으면 "" 반환, strip() 앞뒤 공백 제거
    answer_text = answer.get("answer", "").strip()
else: # answer가 dict 타입이 아닌 경우 그냥 문자열로 변환해서 공백을 제거
    answer_text = str(answer).strip()

# 답변을 못 찾을 경우: "[CLS]" 존재하거나, answer_text 데이터가 없으면, 검색결과 context 사용
if "[CLS]" in answer_text or not answer_text:
    answer_text = context

print("질문:", query)
print("답변:", answer)

# 메모리 정리
# del 객체 삭제
del qa_tokenizer
del qa_model

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

PyTorch Version: 2.6.0, Device: cuda


2026-03-16 11:03:28,804 [INFO] HTTP Request: HEAD https://huggingface.co/monologg/koelectra-base-v3-finetuned-korquad/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:03:28,818 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/monologg/koelectra-base-v3-finetuned-korquad/ad4097ba1dc9904a22c3df71cafe6431e23f8914/config.json "HTTP/1.1 200 OK"
2026-03-16 11:03:29,256 [INFO] HTTP Request: HEAD https://huggingface.co/monologg/koelectra-base-v3-finetuned-korquad/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:03:29,275 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/monologg/koelectra-base-v3-finetuned-korquad/ad4097ba1dc9904a22c3df71cafe6431e23f8914/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-16 11:03:29,578 [INFO] HTTP Request: GET https://huggingface.co/api/models/monologg/koelectra-base-v3-finetuned-korquad/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ElectraForQuestionAnswering LOAD REPORT from: monologg/koelectra-base-v3-finetuned-korquad
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-16 11:03:33,315 [INFO] HTTP Request: POST http://localhost:6333/collections/news_collection/points/query "HTTP/1.1 200 OK"


ID=25, Score=0.78133774, Title=의료 AI 활용, Content=병원에서는 AI 기반 진단 시스템을 도입하여 환자의 영상 데이터를 분석하고 질병을 조기에 발견하는 데 활용하고 있다. 특히 암 진단, 심혈관 질환 예측 등에서 높은 정확도를 보여 의료 현장의 혁신을 이끌고 있다.
ID=21, Score=0.76919353, Title=AI 개요, Content=인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고 있다.
ID=29, Score=0.5149973, Title=기술 혁신 사례, Content=국내 스타트업은 새로운 로봇 기술을 공개하며 제조업과 물류 산업에서의 자동화 가능성을 제시하였다. 이 기술은 인공지능과 센서 융합을 통해 사람과 협업할 수 있는 차세대 로봇을 구현하였다.
질문: AI 의료 활용 사례
답변: 병원에서는 AI 기반 진단 시스템


In [25]:
# 요약 토크나이저 + 요약 모델
# from transformers import BartTokenizer # 영어 전용이라 맞지 않음
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration
from transformers import pipeline
import gc

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# 요약 토크나이저 + 요약 모델 로드 , 요약 모델: 특정 도메인으로 파인튜닝된 QA 모델로 교체 검토
summarizer_model_name = "gogamza/kobart-summarization"
summarizer_tokenizer = PreTrainedTokenizerFast.from_pretrained(summarizer_model_name)
summarizer_model = BartForConditionalGeneration.from_pretrained(summarizer_model_name).to(device)

# 요약 함수 생성
def summarize_text(context, max_length=50, min_length=10):
    # 요약 토크나이저
    inputs = summarizer_tokenizer(
        [context],
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # 요약 생성
    summary_ids = summarizer_model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True,
        
        no_repeat_ngram_size=3,
        repetition_penalty=2.5,   # 반복 억제 강화
        length_penalty=1.5, # 문장 길이 조절

        do_sample=True,
        temperature=0.9, # 다양성 확보
        top_p=0.85 # 안정성 확보
    )
    
    # 요약 Decode 한글로 변환
    summary = summarizer_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return clean_summary(summary)

# 후처리 로직: 요약 결과 문자열에서 바로 앞 단어가 반복되는 경우 제거
def clean_summary(summary: str) -> str:
    # 같은 구절 반복 제거
    tokens = summary.split()          # 1. 문자열을 공백 기준으로 단어 단위 분리
    cleaned = []                      # 2. 최종 결과를 담을 리스트 생성
    for t in tokens:                  # 3. 분리된 단어들을 순서대로 확인
        if not cleaned or cleaned[-1] != t:  
            # 4. 리스트가 비어있거나, 마지막에 추가된 단어와 현재 단어가 다르면
            cleaned.append(t)         #    현재 단어를 결과 리스트에 추가
    return " ".join(cleaned)          # 5. 중복 제거된 단어들을 다시 문자열로 합침
 
# 요약: 검색문서들을 요약
summary = summarize_text(context, max_length=60, min_length=20)
# print("\n요약 결과:", summary)

# 검색+QA_요약: 검색 질의문 + QA 답변 + 검색 결과 -> 요약한다
combined_input = f"질문: {query}\n답변: {answer}\n문서: {context}"
qa_summary = summarize_text(combined_input, max_length=60, min_length=20)
# print("\nQA 입력값:", combined_input)
# print("\nQA+요약 결과:", qa_summary)

# 출처 정보 포함
sources = [ f"ID={r.id}, 제목={r.payload['title']}" for r in results.points ]
final_answer = f"{qa_summary}\n\n출처: " + "; ".join(sources)
print("\n최종 답변:", final_answer)

# 메모리 정리
# del 객체 삭제
del summarizer_tokenizer
del summarizer_model

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

PyTorch Version: 2.6.0, Device: cuda


2026-03-16 11:26:12,351 [INFO] HTTP Request: HEAD https://huggingface.co/gogamza/kobart-summarization/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-03-16 11:26:12,563 [INFO] HTTP Request: GET https://huggingface.co/api/models/gogamza/kobart-summarization/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-16 11:26:12,771 [INFO] HTTP Request: GET https://huggingface.co/api/models/gogamza/kobart-summarization/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-16 11:26:12,981 [INFO] HTTP Request: HEAD https://huggingface.co/gogamza/kobart-summarization/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:26:12,994 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gogamza/kobart-summarization/31f181b155a0ad74bd93bd90ee04310ff72691f4/tokenizer.json "HTTP/1.1 200 OK"
2026-03-16 11:26:13,323 [INFO] HTTP Request: HEAD https://huggingface.co/gogamza/kobart-summarization/r

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

2026-03-16 11:26:14,103 [INFO] HTTP Request: HEAD https://huggingface.co/gogamza/kobart-summarization/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-03-16 11:26:14,312 [INFO] HTTP Request: HEAD https://huggingface.co/gogamza/kobart-summarization/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:26:14,323 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/gogamza/kobart-summarization/31f181b155a0ad74bd93bd90ee04310ff72691f4/config.json "HTTP/1.1 200 OK"



최종 답변: 인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고

출처: ID=25, 제목=의료 AI 활용; ID=21, 제목=AI 개요; ID=29, 제목=기술 혁신 사례


In [27]:
import torch
import gc
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

model_name = "EleutherAI/polyglot-ko-1.3b"
llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder="offload"
).to(device)

prompts = [
    final_answer
]

# 정규식 후처리 함수
def clean_result(text: str) -> str:
    patterns = [
        r"http\S+",              # URL 제거
        r"※.*",                  # 저작권 문구 제거
        r"Q\..*",                # Q&A 스타일 제거
        r"▶.*",                  # ▶ 시작 문구 제거
        r"['\"].*['\"]",         # 따옴표 안 문장 제거
        r"\[사진 출처.*?\]",     # 사진 출처 제거
        r"\[저작권자.*?\]",      # 저작권자 문구 제거
        r"[가-힣]+\s?기자.*?\(.*?\)", # 기자 이름 + 이메일 제거
        r"\S+@\S+",              # 이메일 제거
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text)
    return text.strip()

for i, prompt in enumerate(prompts, 1):
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(device)
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.5,
        top_p=0.8,
        pad_token_id=llm_tokenizer.eos_token_id
    )
    result = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "출력:" in result:
        result = result.split("출력:")[-1].strip()
    result = clean_result(result)

    print(f"=== 결과 {i} ===")
    print("검색 결과:", prompt)
    print("llm 결과:", result)

# 메모리 정리
# del 객체 삭제
del llm_model
del llm_tokenizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

PyTorch Version: 2.6.0, Device: cuda


2026-03-16 11:58:20,419 [INFO] HTTP Request: HEAD https://huggingface.co/EleutherAI/polyglot-ko-1.3b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:58:20,434 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EleutherAI/polyglot-ko-1.3b/557e162cf6e944fdbae05bab2e45d066a125eacb/config.json "HTTP/1.1 200 OK"
2026-03-16 11:58:20,647 [INFO] HTTP Request: HEAD https://huggingface.co/EleutherAI/polyglot-ko-1.3b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:58:20,660 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EleutherAI/polyglot-ko-1.3b/557e162cf6e944fdbae05bab2e45d066a125eacb/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-16 11:58:20,876 [INFO] HTTP Request: HEAD https://huggingface.co/EleutherAI/polyglot-ko-1.3b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:58:20,883 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/model

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

2026-03-16 11:58:24,167 [INFO] HTTP Request: HEAD https://huggingface.co/EleutherAI/polyglot-ko-1.3b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-16 11:58:24,180 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EleutherAI/polyglot-ko-1.3b/557e162cf6e944fdbae05bab2e45d066a125eacb/generation_config.json "HTTP/1.1 200 OK"


=== 결과 1 ===
검색 결과: 인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고

출처: ID=25, 제목=의료 AI 활용; ID=21, 제목=AI 개요; ID=29, 제목=기술 혁신 사례
llm 결과: 인공지능(AI)은 인간의 학습, 추론, 문제 해결 능력을 모방하기 위해 개발된 기술로, 머신러닝과 딥러닝을 포함한 다양한 알고리즘을 활용한다. 최근에는 자연어 처리, 이미지 인식, 자율주행 등 여러 분야에서 활용되고 있으며, 산업 전반에 걸쳐 혁신을 이끌고

출처: ID=25, 제목=의료 AI 활용; ID=21, 제목=AI 개요; ID=29, 제목=기술 혁신 사례; ID=39, 제목=AI 활용 사례; ID=49, 제목=AI 활용 사례; ID=50, 제목=AI 개요; ID=58, 제목=의료 AI 활용; ID=59, 제목=AI 개요; ID=60, 제목=의료 AI 활용; ID=61
